006 DATA CLEANING

- Start with 005_data_only_pedestrians parquet file
- End with 006_data_only_pedestrians parquet file

OBJECTIVES: 
- Resolve any issues with 'DataOraIncidente' values
- Detect and delete rows with no time
- Apply phases of day to every accident
- Split year, month, date, day of week for easier analysis

In [36]:
import pandas as pd
import numpy as np
import pyarrow

In [37]:
df = pd.read_parquet("005_data_only_pedestrians.parquet").copy()
df.shape

(19713, 43)

The dataset currently has 43 columns and 19713 rows of data. Each row now refers to a pedestrian. 

In [38]:
df.columns

Index(['Protocollo', 'TipoVeicolo', 'StatoVeicolo', 'NaturaIncidente',
       'NUM_FERITI', 'NUM_RISERVATA', 'NUM_MORTI', 'NUM_ILLESI', 'Sesso',
       'DataOraIncidente', 'CondizioneAtmosferica', 'Traffico', 'Gruppo',
       'Localizzazione1', 'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02',
       'Chilometrica', 'DaSpecificare', 'Latitude', 'Longitude',
       'particolaritastrade', 'TipoStrada', 'FondoStradale', 'Pavimentazione',
       'Segnaletica', 'Visibilita', 'Illuminazione', 'Tipolesione', 'Deceduto',
       'DecedutoDopo', 'parked_vehicle_involved', 'driver_injury',
       'driver_deceduto', 'driver_deceduto_dopo', 'driver_Sesso',
       'passenger_no_of_females', 'passenger_no_of_males',
       'passenger_no_of_unknown_sex', 'passengers_killed',
       'passengers_uninjured', 'passengers_injured'],
      dtype='object')

In [39]:
def analyse_column(column_name):
    """
    Analyzes value counts, percentages, and unique protocols for a column in the DataFrame.

    Args:
        column_name (str): The name of the column to analyze.

    Returns:
        pandas.DataFrame: A DataFrame with 'Count', '% per category', and 
                         'Unique Accidents' for each unique value.
    """

    # Validate input
    if column_name not in df.columns:
        raise ValueError(f"Column '{column_name}' not found in DataFrame")

    # Clean the column by replacing empty strings with NaN
    cleaned_column = df[column_name].replace('', pd.NA)

    # Calculate counts and percentages
    counts = cleaned_column.value_counts(dropna=False)
    percentages = (cleaned_column.value_counts(
        dropna=False, normalize=True) * 100).round(2)

    # Calculate unique protocols for each category
    unique_protocols = []
    for category in counts.index:
        if pd.isna(category):
            # Handle NaN values
            mask = cleaned_column.isna()
        else:
            # Handle non-NaN values
            mask = cleaned_column == category

        protocol_count = df.loc[mask, 'Protocollo'].nunique()
        unique_protocols.append(protocol_count)

    # Create result DataFrame
    result = pd.DataFrame({
        'Count': counts,
        '% per category': percentages,
        'Unique Accidents': unique_protocols
    }, index=counts.index)

    return result

CLEANING THE 'DATE AND TIME OF ACCIDENT' COLUMN

First we identify rows where the 'DataOraIncidente' is blank:

In [40]:
df['DataOraIncidente'].dtype

dtype('O')

In [41]:
df[df['DataOraIncidente'] == '']

,Protocollo,TipoVeicolo,StatoVeicolo,NaturaIncidente,NUM_FERITI,NUM_RISERVATA,NUM_MORTI,NUM_ILLESI,Sesso,DataOraIncidente,...,driver_injury,driver_deceduto,driver_deceduto_dopo,driver_Sesso,passenger_no_of_females,passenger_no_of_males,passenger_no_of_unknown_sex,passengers_killed,passengers_uninjured,passengers_injured


In [42]:
df[df['DataOraIncidente'].isna()]

,Protocollo,TipoVeicolo,StatoVeicolo,NaturaIncidente,NUM_FERITI,NUM_RISERVATA,NUM_MORTI,NUM_ILLESI,Sesso,DataOraIncidente,...,driver_injury,driver_deceduto,driver_deceduto_dopo,driver_Sesso,passenger_no_of_females,passenger_no_of_males,passenger_no_of_unknown_sex,passengers_killed,passengers_uninjured,passengers_injured


There are no rows where there is a missing value in the 'DataOraIncidente' column. 

Now we search for rows where the time is missing. The time is fundamental for an analysis of twilight, so any accidents with the time missing will need to be dropped. 

In [43]:
no_timestamp = df[~df['DataOraIncidente'].str.contains(':', na=False)]
print(f"Number of rows without timestamp: {len(no_timestamp)}")
print(
    f"Number of unique protocols affected: {no_timestamp['Protocollo'].nunique()}")

Number of rows without timestamp: 26
Number of unique protocols affected: 23


In [44]:
no_timestamp = df[~df['DataOraIncidente'].str.contains(':', na=False)]
no_timestamp_rows = no_timestamp.index.tolist()
print(no_timestamp_rows)

[599, 1589, 2522, 2803, 4809, 4937, 7466, 8811, 9094, 9095, 9522, 10123, 10124, 10566, 11339, 11967, 13968, 14029, 14257, 14550, 14551, 14699, 15632, 17060, 18347, 18855]


Let's examine the rows with no timestamp:

In [45]:
df.iloc[no_timestamp_rows, :4]

,Protocollo,TipoVeicolo,StatoVeicolo,NaturaIncidente
599,1940690,Motociclo a solo,In marcia / fermata / arresto,Investimento di pedone
1589,1955843,Unknown,Allontanatosi,Investimento di pedone
2522,1999297,Autovettura di polizia,In marcia / fermata / arresto,Investimento di pedone
2803,2031895,Quadriciclo leggero,In marcia / fermata / arresto,Investimento di pedone
4809,2321496,Autovettura privata,In marcia / fermata / arresto,Investimento di pedone
4937,2346090,Autovettura privata,In marcia / fermata / arresto,Investimento di pedone
7466,2898909,Autovettura privata,In marcia / fermata / arresto,Investimento di pedone
8811,3216043,Unknown,Allontanatosi,Investimento di pedone
9094,3283218,Autovettura privata,In marcia / fermata / arresto,Investimento di pedone
9095,3283218,Autovettura privata,In marcia / fermata / arresto,Investimento di pedone


Some of these accidents were hit and run (vehicle: unknown, vehicle state: allontanatosi, that is, leaving the scene), so if the pedestrian is injured, they may not have been able to report a time. 

Other accidents simply have no timestamp.

23 separate accidents are documented, involving a total of 26 pedestrians. 
Due to the importance of time of day in visibility analysis, we will exclude these 26 rows with missing time data.
This subset was too small (26 out of 19,713 = 0.13%, which is statistically negligible) to warrant synthetic imputation, which carries the risk of introducing bias.

In [46]:
df = df.drop(no_timestamp_rows).reset_index(drop=True)
df.shape

(19687, 43)

The number of rows has gone from 19,713 to 19,687, down by 26, as expected. 

For each date and time, we want to assign one of ten phases of day:

- time_night_am               midnight to astronomical dawn   (midnight to depression of 18 degrees)
- time_astronomical_dawn                                      (depression of 18 to 12 degrees)
- time_nautical_dawn                                          (depression of 12 to 6 degrees)
- time_civil_dawn                                             (depression of 6 to 0 degrees)
- time_daylight_am                                            (sunrise to sun at 90 degrees)
- time_daylight_pm                                            (sun at 90 degrees to sunset)
- time_civil_dusk                                             (depression of 0 to 6 degrees)
- time_nautical_dusk                                          (depression of 6 to 12 degrees)
- time_astronomical_dusk                                      (depression of 12 to 18 degrees)
- time_night_pm               astronomical dusk to midnight   (depression of 18 degrees to midnight)

We will then apply one hot encoding to each of these 10 categories. 

Now we ensure the 'DataOraIncidente' column timestamps are of the datetime format, so that we can then apply pandas and astral functions to them without any reading issues. 

In [47]:
df['DataOraIncidente'] = pd.to_datetime(
    df['DataOraIncidente'], format='mixed', dayfirst=True, errors='raise')

In [48]:
import pytz

In [49]:
# Now identify ambiguous times BEFORE localizing
# Get timezone naive DatetimeIndex
dt_naive = pd.DatetimeIndex(df['DataOraIncidente'])

# Use pandas built-in function to find ambiguous datetimes
ambiguous_mask = pd.Series(pd.Series(dt_naive).map(
    lambda x: pd.Timestamp(x).tz_localize(
        'Europe/Rome', ambiguous='NaT', nonexistent='shift_forward')
    is pd.NaT
))

# Count how many ambiguous
num_ambiguous = ambiguous_mask.sum()
print(f"Found {num_ambiguous} ambiguous timestamps (DST overlap).")

Found 1 ambiguous timestamps (DST overlap).


In [50]:
# Apply localization: always pick winter time for ambiguous
df['DataOraIncidente'] = df['DataOraIncidente'].dt.tz_localize(
    'Europe/Rome',
    # Always choose winter time version (conservative for accidents at night)
    ambiguous=False,
    nonexistent='shift_forward'  # Handle spring forward safely
)

In [51]:
from astral import LocationInfo
from astral.sun import dawn, sunrise, sunset, dusk, noon, sun
from datetime import datetime, date, time, timedelta

In [52]:
# Define location for Rome
rome = LocationInfo("Rome", "Italy", "Europe/Rome", 41.9028, 12.4964)

In [53]:
# Function to classify the phase of day
def classify_phase(ts):
    observer = rome.observer
    date = ts.date()

    # Cut off times
    civil_dawn = dawn(observer, date, depression=6)
    nautical_dawn = dawn(observer, date, depression=12)
    astronomical_dawn = dawn(observer, date, depression=18)

    sunrise_time = sunrise(observer, date)
    noon_time = noon(observer, date)
    sunset_time = sunset(observer, date)

    civil_dusk = dusk(observer, date, depression=6)
    nautical_dusk = dusk(observer, date, depression=12)
    astronomical_dusk = dusk(observer, date, depression=18)

    # Classifications
    if ts < astronomical_dawn:
        return "time_night_am"
    elif ts < nautical_dawn:
        return "time_astronomical_twilight_am"
    elif ts < civil_dawn:
        return "time_nautical_twilight_am"
    elif ts < sunrise_time:
        return "time_civil_twilight_am"
    elif ts < noon_time:
        return "time_daylight_am"
    elif ts < sunset_time:
        return "time_daylight_pm"
    elif ts < civil_dusk:
        return "time_civil_twilight_pm"
    elif ts < nautical_dusk:
        return "time_nautical_twilight_pm"
    elif ts < astronomical_dusk:
        return "time_astronomical_twilight_pm"
    else:
        return "time_night_pm"

In [54]:
# Apply to dataframe
df['phase_of_day'] = df['DataOraIncidente'].apply(classify_phase)

In [55]:
df['phase_of_day'].value_counts(dropna=False)

phase_of_day
time_daylight_am                 7025
time_daylight_pm                 6525
time_night_pm                    2515
time_nautical_twilight_pm        1092
time_astronomical_twilight_pm     902
time_civil_twilight_pm            720
time_night_am                     501
time_civil_twilight_am            164
time_nautical_twilight_am         157
time_astronomical_twilight_am      86
Name: count, dtype: int64

In [56]:
df['phase_of_day'].value_counts().sum()

np.int64(19687)

In [57]:
df.shape

(19687, 44)

Now there are 43 + 1 = 44 columns; the new column, 'phase_of_day', assigns a phase of day (from midnight to dawn, nautical /  astronomical / civil twilight, daylight before sunpeak, daylight after sunpeak, dusk to midnight) to each the accident. 

In order to look for further patterns, we also split the datetimestamp into years, months, dates and days of the week:
The column 'DAY' is encoded: Monday = 1, Tuesday = 2, etc

In [58]:
df['YEAR'] = df['DataOraIncidente'].dt.year
df['MONTH'] = df['DataOraIncidente'].dt.month
df['DATE'] = df['DataOraIncidente'].dt.day
df['TIME'] = df['DataOraIncidente'].dt.time
df['DAY'] = df['DataOraIncidente'].dt.dayofweek + 1

Now we should have 19,687 rows and 44 + 5 (year, month, date, day, time) = 49

In [59]:
df.shape

(19687, 49)

In [60]:
df.to_parquet('006_data_only_pedestrians.parquet', index=False)

In [61]:
df.to_csv('006_data_only_pedestrians.csv', index=False)

#### TWILIGHT DATAFRAME

Now we will make a separate dataframe which will give us, day by day, the length of each phase of day in hours, enabling us to calculate an accidents / hour and injured persons / hour. 

In [62]:
# --- Twilight/day-phase feature table for Rome (2013-01-01 → 2022-08-31) ---
# Location & TZ
tz = pytz.timezone("Europe/Rome")

# Date range
start = pd.Timestamp("2013-01-01").date()
end = pd.Timestamp("2022-08-31").date()
dates = pd.date_range(start=start, end=end, freq="D")


def hours(a, b):
    """Duration in hours between two aware datetimes (never negative)."""
    return max(0.0, (b - a).total_seconds() / 3600.0)


rows = []
for dts in dates:
    d = dts.date()

    # Local midnight boundaries for that calendar day
    midnight_start = tz.localize(datetime.combine(d, time(0, 0, 0)))
    midnight_end = tz.normalize(
        midnight_start + timedelta(days=1))  # <- normalize

    # Morning twilight chain (dawn with different depressions)
    astro_dawn = dawn(rome.observer, date=d, tzinfo=tz,
                      depression=18)  # astronomical
    naut_dawn = dawn(rome.observer, date=d, tzinfo=tz,
                     depression=12)  # nautical
    civil_dawn = dawn(rome.observer, date=d, tzinfo=tz, depression=6)   # civil
    srise = sunrise(rome.observer, date=d, tzinfo=tz)

    # Noon & evening
    sdict = sun(rome.observer, date=d, tzinfo=tz)
    solar_noon = sdict["noon"]
    sset = sunset(rome.observer, date=d, tzinfo=tz)
    civil_dusk = dusk(rome.observer, date=d, tzinfo=tz, depression=6)
    naut_dusk = dusk(rome.observer, date=d, tzinfo=tz, depression=12)
    astro_dusk = dusk(rome.observer, date=d, tzinfo=tz, depression=18)

    # Build 10 phases (hours)
    phase_01 = hours(midnight_start, astro_dawn)   # night -> astronomical dawn
    phase_02 = hours(astro_dawn, naut_dawn)        # AT am
    phase_03 = hours(naut_dawn, civil_dawn)        # NT am
    phase_04 = hours(civil_dawn, srise)            # CT am
    phase_05 = hours(srise, solar_noon)            # day am
    phase_06 = hours(solar_noon, sset)             # day pm
    phase_07 = hours(sset, civil_dusk)             # CT pm
    phase_08 = hours(civil_dusk, naut_dusk)        # NT pm
    phase_09 = hours(naut_dusk, astro_dusk)        # AT pm
    phase_10 = hours(astro_dusk, midnight_end)     # night pm

    # Tiny numerical tidy: force sum to exactly 24h if off by a hair
    total = sum([phase_01, phase_02, phase_03, phase_04, phase_05,
                 phase_06, phase_07, phase_08, phase_09, phase_10])
    if abs(total - 24.0) > 1e-6:
        phase_10 += (24.0 - total)

    rows.append({
        "date": d,
        "phase_01_night_am": phase_01,
        "phase_02_AT_am":    phase_02,
        "phase_03_NT_am":    phase_03,
        "phase_04_CT_am":    phase_04,
        "phase_05_day_am":   phase_05,
        "phase_06_day_pm":   phase_06,
        "phase_07_CT_pm":    phase_07,
        "phase_08_NT_pm":    phase_08,
        "phase_09_AT_pm":    phase_09,
        "phase_10_night_pm": phase_10,
    })

df_twilight = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)

# Quick sanity check: every row should sum to ~24 hours
phase_hour_cols = [c for c in df_twilight.columns
                   if c.startswith("phase_") and not c.endswith("_min")]
bad = df_twilight.loc[~np.isclose(df_twilight[phase_hour_cols].sum(axis=1), 24, atol=1e-6),
                      ["date"] + phase_hour_cols]
bad.head()

,date,phase_01_night_am,phase_02_AT_am,phase_03_NT_am,phase_04_CT_am,phase_05_day_am,phase_06_day_pm,phase_07_CT_pm,phase_08_NT_pm,phase_09_AT_pm,phase_10_night_pm


In [63]:
df_twilight.head()

,date,phase_01_night_am,phase_02_AT_am,phase_03_NT_am,phase_04_CT_am,phase_05_day_am,phase_06_day_pm,phase_07_CT_pm,phase_08_NT_pm,phase_09_AT_pm,phase_10_night_pm
0,2013-01-01,5.955092,0.563267,0.580949,0.535393,4.589188,4.599466,0.535421,0.580993,0.563328,5.496904
1,2013-01-02,5.957817,0.562888,0.580426,0.534760,4.596053,4.605973,0.534792,0.580474,0.562955,5.483861
2,2013-01-03,5.960009,0.562480,0.579862,0.534080,4.603014,4.613490,0.534114,0.579915,0.562554,5.470482
3,2013-01-04,5.961662,0.562045,0.579259,0.533352,4.610905,4.621168,0.533390,0.579317,0.562126,5.456777
4,2013-01-05,5.962771,0.561583,0.578618,0.532578,4.619172,4.629547,0.532620,0.578682,0.561671,5.442758


In [64]:
# Save for reuse
df_twilight.to_parquet("df_twilight.parquet", index=False)
df_twilight.to_csv("df_twilight.csv", index=False)

In [ ]:
metadata = {
    'notebook': '006 Data cleaning times and phases.ipynb',
    'step': 'time normalization, twilight phases, phase_of_day + cyclical encodings',

    # Inputs/outputs
    'initial_parquet_file': '005_data_only_pedestrians.parquet',
    'final_parquet_file': '006_times_phases.parquet',

    # Shapes
    'initial_rows': 19_713,
    'initial_columns': 43,
    'final_rows': 19_713,   # feature engineering only; no row filtering expected
    'final_columns': 43 + __import__("builtins").len([
        'timestamp_local', 'hour', 'weekday', 'is_weekend', 'month', 'doy',
                                   'time_sin', 'time_cos', 'doy_sin', 'doy_cos',
                                   'phase_of_day', 'lighting_insufficient', 'twilight_phase',
                                   'sunrise_time', 'sunset_time', 'civil_dawn', 'civil_dusk',
                                   'nautical_dawn', 'nautical_dusk', 'astronomical_dawn', 'astronomical_dusk',
                                   'holiday_italy'
    ]),  # adjust if you added fewer/more

    # Columns touched
    'columns_added': [
        # temporal basics
        'timestamp_local', 'hour', 'weekday', 'is_weekend', 'month', 'doy',
        # cyclical encodings
        'time_sin', 'time_cos', 'doy_sin', 'doy_cos',
        # phases & lighting
        'phase_of_day', 'lighting_insufficient', 'twilight_phase',
        # solar events (if computed via Astral)
        'sunrise_time', 'sunset_time', 'civil_dawn', 'civil_dusk',
        'nautical_dawn', 'nautical_dusk', 'astronomical_dawn', 'astronomical_dusk',
        # calendar features
        'holiday_italy'
    ],
    'columns_removed': [],
    'columns_modified': [
        # e.g. parsed/standardized; keep generic to match your actual names
        'original_datetime_columns_parsed_to_timestamp_local'
    ],

    # Timezone & calendars
    'timezone_used': 'Europe/Rome',
    'dst_handling': 'localized to Europe/Rome (aware datetimes)',
    'holiday_calendar': 'Italy (holidays.IT)',

    # Solar/twilight computation
    'phases_library': 'astral',
    'twilight_definition': 'civil (default), with nautical/astronomical if computed',
    'latlon_source': 'row-wise accident coordinates (Latitude/Longitude)',
    'assumed_elevation_m': 0,

    # Encodings & mapping used
    'cyclical_encodings': {
        'time_sin': 'sin(2π * hour/24)',   'time_cos': 'cos(2π * hour/24)',
        'doy_sin': 'sin(2π * day_of_year/365)', 'doy_cos': 'cos(2π * day_of_year/365)'
    },
    'phase_of_day_mapping': {
        'night': 'after civil_dusk or before civil_dawn',
        'twilight': 'between civil_dawn–sunrise or sunset–civil_dusk',
        'day': 'between sunrise and sunset'
    },
    'lighting_rule': "lighting_insufficient = (phase_of_day != 'day')",

    # QA & data hygiene
    'qa_checks': {
        'timestamp_na_before': '<fill>',
        'timestamp_na_after': '<fill>',
        'rows_missing_latlon': '<fill>',
        'solar_calc_failures': '<fill>',
        'tz_aware_datetimes': True,
        'encodings_finite': True
    },

    # Decisions captured
    'decisions_made': [
        'Standardized timestamps and localized to Europe/Rome with DST awareness.',
        'Derived phase_of_day via Astral solar events using per-row coordinates.',
        'Added cyclical encodings for hour and day-of-year to capture periodicity.',
        'Flagged lighting_insufficient for non-daylight conditions.',
        'Annotated Italian public holidays and is_weekend for calendar effects.',
        'Left row count unchanged to avoid accidental filtering in feature step.'
    ],

    # Provenance (optional—fill if available)
    'run_timestamp': '<auto-fill: pd.Timestamp.now(tz=\"Europe/Rome\").isoformat()>',
    'git_commit': '<fill if repo is git-tracked>',
    'library_versions': {
        'python': '<fill>',
        'pandas': '<fill>',
        'astral': '<fill>',
        'pyarrow': '<fill>'
    }
}